In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/bmw_pricing_challenge - bmw_pricing_challenge.csv')
print('Shape:', df.shape)
df.head()

Shape: (4843, 18)


,maker_key,model_key,mileage,engine_power,registration_date,fuel,paint_color,car_type,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,price,sold_at
0,BMW,118,140411,100,01-02-2012,diesel,black,convertible,True,True,False,False,True,True,True,False,11300,01-01-2018
1,BMW,M4,13929,317,01-04-2016,petrol,grey,convertible,True,True,False,False,False,True,True,True,69700,01-02-2018
2,BMW,320,183297,120,01-04-2012,diesel,white,convertible,False,False,False,False,True,False,True,False,10200,01-02-2018
3,BMW,420,128035,135,01-07-2014,diesel,red,convertible,True,True,False,False,True,True,True,True,25100,01-02-2018
4,BMW,425,97097,160,01-12-2014,diesel,silver,convertible,True,True,False,False,False,True,True,True,33400,01-04-2018


In [ ]:
# First Audit
print('===BASIC INFO===')
print('Rows', df.shape[0])
print('Columns', df.shape[1])
print('\n===Data Types===')
print(df.dtypes)
print('\n===COLUMN TYPE COUNTS===')
print('Numerical Columns Count:', len(df.select_dtypes('number').columns.to_list()))
print('Categorical Columns Count:', len(df.select_dtypes('str').columns.to_list()))

===BASIC INFO===
Rows 4843
Columns 18

===Data Types===
maker_key              str
model_key              str
mileage              int64
engine_power         int64
registration_date      str
fuel                   str
paint_color            str
car_type               str
feature_1             bool
feature_2             bool
feature_3             bool
feature_4             bool
feature_5             bool
feature_6             bool
feature_7             bool
feature_8             bool
price                int64
sold_at                str
dtype: object

===COLUMN TYPE COUNTS===
Numerical Columns Count: 3
Categorical Columns Count: 7


In [ ]:
# Categorical Columns audit

print('=== UNIQUE VALUES PER CATEGORICAL COLUMN ===')
for col in df.select_dtypes('str').columns:
    print(f'{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(5))
    print()

=== UNIQUE VALUES PER CATEGORICAL COLUMN ===
maker_key: 1 unique values
maker_key
BMW    4843
Name: count, dtype: int64

model_key: 75 unique values
model_key
320    752
520    633
318    569
X3     438
116    358
Name: count, dtype: int64

registration_date: 199 unique values
registration_date
01-07-2013    173
01-03-2014    162
01-05-2014    153
01-01-2013    148
01-09-2013    148
Name: count, dtype: int64

fuel: 4 unique values
fuel
diesel           4641
petrol            191
hybrid_petrol       8
electro             3
Name: count, dtype: int64

paint_color: 10 unique values
paint_color
black    1633
grey     1175
blue      710
white     538
brown     341
Name: count, dtype: int64

car_type: 8 unique values
car_type
estate        1606
sedan         1168
suv           1058
hatchback      699
subcompact     117
Name: count, dtype: int64

sold_at: 9 unique values
sold_at
01-05-2018    809
01-03-2018    739
01-04-2018    693
01-06-2018    604
01-07-2018    537
Name: count, dtype: int64


In [ ]:
# Numerical Columns audit

print('=== NUMERICAL STATISTICS ===')
df.describe().T

=== NUMERICAL STATISTICS ===


,count,mean,std,min,25%,50%,75%,max
mileage,4843.0,140962.799504,60196.740703,-64.0,102913.5,141080.0,175195.5,1000376.0
engine_power,4843.0,128.988230,38.993360,0.0,100.0,120.0,135.0,423.0
price,4843.0,15828.081767,9220.285684,100.0,10800.0,14200.0,18600.0,178500.0


In [ ]:
# Target Columns Audit

print('===TARGET COLUMN===')
df['price'].describe()

===TARGET COLUMN===


count      4843.000000
mean      15828.081767
std        9220.285684
min         100.000000
25%       10800.000000
50%       14200.000000
75%       18600.000000
max      178500.000000
Name: price, dtype: float64

In [ ]:
# Checking for missing values

# Universal missing value scanner
missing_disguises = ['?', 'NA', 'N/A', 'na', 'none', 
                     'None', 'NONE', '-', '--', 'unknown', ' ', '']

print('=== DISGUISED MISSING VALUES ===')
for col in df.select_dtypes('str').columns:
    for disguise in missing_disguises:
        count = (df[col] == disguise).sum()
        if count > 0:
            print(f'{col} : "{disguise}" appears {count} times')

print('\n=== ACTUAL NULL VALUES ===')
missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100
missing_df = pd.DataFrame({
    'count': missing,
    'percentage': missing_pct
})
missing_df = missing_df[missing_df['count'] > 0].sort_values(
    by='percentage', ascending=False
)
print(missing_df)

=== DISGUISED MISSING VALUES ===

=== ACTUAL NULL VALUES ===
Empty DataFrame
Columns: [count, percentage]
Index: []


In [ ]:
# checking for duplicates

print('Number of dupicated Columns:', df.duplicated().sum())

Number of dupicated Columns: 0


# Cleaning Strategy

- The maker_key has only one unique value so will drop it
- The model_key column has 75 unique values, too many for categorical, need more information on how to handle this column
- The registration_date column type is str, need to convert that to datetime format
- The fuel type column has 4 unique values but majority of the samples(96%) are from diesel and the rest together accounts to only 4% of the data. Need to look for feature engineer or else drop the column.
- The paint columns has 10 unique values, based on the data distribution, will feature engineer it to have 3 unique values - common, standard and rare colors.
- Creating a Column for registration year and Sold_at year and dropping the registration date and sold_at date columns
- The car _type column 4 significant sample size but 4 car types has a very few sample, will keep as it is right now, if required will feature engineer later
- The sold_at column is a date type but saved as str so will convert that to datetime format.
- The mileage column has negative minimum value which is not possible, need to investigate later to fix the error.
- The engine_power column has a value that contains 0 which seems like an error, need to investigate more on it.
- The price column seems to be skewed, need to fix that before modeling

In [ ]:
# Dropping the maker_key column

df = df.drop(columns='maker_key')

df.head()

,model_key,mileage,engine_power,registration_date,fuel,paint_color,car_type,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,price,sold_at
0,118,140411,100,01-02-2012,diesel,black,convertible,True,True,False,False,True,True,True,False,11300,01-01-2018
1,M4,13929,317,01-04-2016,petrol,grey,convertible,True,True,False,False,False,True,True,True,69700,01-02-2018
2,320,183297,120,01-04-2012,diesel,white,convertible,False,False,False,False,True,False,True,False,10200,01-02-2018
3,420,128035,135,01-07-2014,diesel,red,convertible,True,True,False,False,True,True,True,True,25100,01-02-2018
4,425,97097,160,01-12-2014,diesel,silver,convertible,True,True,False,False,False,True,True,True,33400,01-04-2018


In [ ]:
# Converting the registration_date and sold_at columns to datetime format

df['registration_date'] = pd.to_datetime(df['registration_date'], errors='coerce')
df['sold_at'] = pd.to_datetime(df['sold_at'], errors='coerce')

df[['registration_date', 'sold_at']].dtypes

np.float64(3.3236070227075594)

In [ ]:
df[df['price']<=100]

,maker_key,model_key,mileage,engine_power,registration_date,fuel,paint_color,car_type,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,price,sold_at
565,BMW,320,179358,120,01-06-2013,diesel,black,estate,False,True,False,False,False,False,True,True,100,01-03-2018
630,BMW,318,147558,105,01-11-2014,diesel,white,estate,False,True,False,False,False,False,False,True,100,01-03-2018
879,BMW,318,134156,105,01-06-2014,diesel,grey,estate,False,True,False,False,False,False,False,True,100,01-04-2018
1255,BMW,320,170381,135,01-07-2013,diesel,silver,estate,True,True,False,False,False,False,True,False,100,01-05-2018
1832,BMW,116,174524,85,01-07-2014,diesel,blue,hatchback,False,True,False,False,False,False,True,True,100,01-03-2018
2829,BMW,525,439060,105,01-10-1996,diesel,silver,sedan,False,False,True,False,False,False,True,False,100,01-03-2018
4356,BMW,X3,79685,190,01-02-2014,diesel,black,suv,False,False,False,False,False,False,False,True,100,01-05-2018


In [ ]:
# dropping the maker_key column

df = df.drop(columns='maker_key')

df.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,price,registration_year,sold_at_year
0,118,140411,100,diesel,black,convertible,1,1,0,0,1,1,1,0,11300,2012,2018
1,M4,13929,317,petrol,grey,convertible,1,1,0,0,0,1,1,1,69700,2016,2018
2,320,183297,120,diesel,white,convertible,0,0,0,0,1,0,1,0,10200,2012,2018
3,420,128035,135,diesel,red,convertible,1,1,0,0,1,1,1,1,25100,2014,2018
4,425,97097,160,diesel,silver,convertible,1,1,0,0,0,1,1,1,33400,2014,2018


In [ ]:
# Imputing the error at engine power

df.loc[df['engine_power']==0, 'engine_power'] = df.loc[df['model_key']=='X1', 'engine_power'].mean().round().astype('int')

print(df['engine_power'].min())

25


In [ ]:
# Imputing the error at mileage

df.loc[df['mileage']<=0, 'mileage'] = df.loc[df['model_key']=='640 Gran Coupé', 'mileage'].mean().round().astype('int')

print(df['mileage'].min())

476


In [ ]:
df.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,price,registration_year,sold_at_year
0,118,140411,100,diesel,black,convertible,1,1,0,0,1,1,1,0,11300,2012,2018
1,M4,13929,317,petrol,grey,convertible,1,1,0,0,0,1,1,1,69700,2016,2018
2,320,183297,120,diesel,white,convertible,0,0,0,0,1,0,1,0,10200,2012,2018
3,420,128035,135,diesel,red,convertible,1,1,0,0,1,1,1,1,25100,2014,2018
4,425,97097,160,diesel,silver,convertible,1,1,0,0,0,1,1,1,33400,2014,2018


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
mileage,4843.0,140986.443733,60163.815962,476.0,102988.5,141080.0,175195.5,1000376.0
engine_power,4843.0,129.010944,38.950223,25.0,100.0,120.0,135.0,423.0
feature_1,4843.0,0.549659,0.497579,0.0,0.0,1.0,1.0,1.0
feature_2,4843.0,0.792690,0.405421,0.0,1.0,1.0,1.0,1.0
feature_3,4843.0,0.201941,0.401490,0.0,0.0,0.0,0.0,1.0
feature_4,4843.0,0.198637,0.399015,0.0,0.0,0.0,0.0,1.0
feature_5,4843.0,0.460458,0.498485,0.0,0.0,0.0,1.0,1.0
feature_6,4843.0,0.241379,0.427964,0.0,0.0,0.0,0.0,1.0
feature_7,4843.0,0.932067,0.251657,0.0,1.0,1.0,1.0,1.0
feature_8,4843.0,0.540987,0.498369,0.0,0.0,1.0,1.0,1.0


In [ ]:
df.to_csv("../data/processed/clean_bmw_price_prediction.csv", index=False)
print('Clean Data Saved')
print(df.shape)

Clean Data Saved
(4843, 17)
